# Naive Bayes for Text Classification

This Notebook is based on 
https://towardsdatascience.com/text-classification-using-naive-bayes-theory-a-working-example-2ef4b7eb7d5a

Please look there for more details.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import MultinomialNB
print('done')

**Counting words in a text**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
corpus = [
    'Earn Money With a Bitcoin Trading Robot. More Bitcoin',
    'We need a new Robot for the Production line',
    'Dear Friend Can we meet For Lunch ?',
    'Dear Rainer, The robot for the Lab has to be payed with bitcoin!']
vectorizer = CountVectorizer()
vectorizer.fit(corpus)  #look at all the word and collect them
print(vectorizer.get_feature_names_out()) #returns the learned vocabulary


In [ ]:
X = vectorizer.transform(corpus)
print(X.toarray())

# There is a shorter version combining fit & transform, but we  
# need it also seperated if we want to classify new texts 
# X = vectorizer.fit_transform(corpus) # shorter

In [ ]:
# original data is sparse 
print(X)

**Pretty printing the vectorized result**

In [ ]:
from pandas import DataFrame

def format_vertical_headers(df):
    """Display a dataframe with vertical column headers"""
    styles = [dict(selector="th", props=[('width', '40px')]),
              dict(selector="th.col_heading",
                   props=[("writing-mode", "vertical-rl"),
                          ('transform', 'rotateZ(180deg)'), 
                          ('height', '40px'),
                          ('vertical-align', 'top')])]
    return (df.fillna('').style.set_table_styles(styles))

df = DataFrame(X.toarray())
df.columns =vectorizer.get_feature_names_out()
display(format_vertical_headers(df))


**Determining Word Frequencies in a text (term frequency–inverse document frequency)**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer()
X= vec.fit_transform(corpus)
print(X)
type(X)
# explanation (document, wordnumber)  TDIDF
# the TF-IDF value increases with every appearance of the word in a document,
# but is gradually decreased with every appearance in other documents. 

In [ ]:
import pandas as pd
df = pd.DataFrame(X.todense())
df.columns =vectorizer.get_feature_names_out()
display(format_vertical_headers(df))

**Loading and splitting serious data - NEWSGROUPS**

In [ ]:
# Load the dataset
# This may take some minutes!
from sklearn.datasets import fetch_20newsgroups
data = fetch_20newsgroups()# Get the text categories may take a while when excuted the dirst time
text_categories = data.target_names# define the training set
train_data = fetch_20newsgroups(subset="train", categories=text_categories)# define the test set
test_data = fetch_20newsgroups(subset="test", categories=text_categories)
print("Done")


**Insepcting the data**

In [ ]:
print("We have {} unique classes".format(len(text_categories)))
print("We have {} training samples".format(len(train_data.data)))
print("We have {} test samples".format(len(test_data.data)))
print("Classes", text_categories)

In [ ]:
print("One Example text:\n", train_data.data[3])
print("Classification:\n", text_categories[train_data.target[3]])

**Testing the Vectorizer**

In [ ]:
vec = CountVectorizer()
transformed_data = vec.fit_transform(train_data.data)
print(transformed_data)
transformed_data.shape

In [ ]:
transformed_data.toarray()[0]

**Composing the model with a pipeline**

In [ ]:
# Build the model
from sklearn.pipeline import make_pipeline
model = make_pipeline(CountVectorizer(), MultinomialNB())# Train the model using the training data
model.fit(train_data.data, train_data.target)# Predict the categories of the test data

In [ ]:
print(len(test_data.data))
#print(test_data.data[0])
predicted_categories = model.predict(test_data.data)
print(predicted_categories)

In [ ]:
#convert category numbers to category texts 
print([text_categories[c] for c in predicted_categories])

In [ ]:
print("Predictions: ", predicted_categories[0:10])
print("Ground Truth Labels: ", test_data.target[0:10])

**Evaluating the model**

In [ ]:
from sklearn.metrics import accuracy_score
print("The accuracy is {}".format(accuracy_score(test_data.target, predicted_categories)))

#Equivalent but repeats making the predictions:
model.score(test_data.data, test_data.target) 

**Vizualizing and Evaluating the model**

In [ ]:
# plot the confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(10,10))
plt.rcParams.update({'font.size': 8})
cm = confusion_matrix(test_data.target, predicted_categories)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
#                               display_labels=model.classes_)
                              display_labels=text_categories)
disp.plot(ax=ax, xticks_rotation='vertical')

